# Baseline models and sanity check

This notebook loads the wide table, shows a few exploratory plots and fits three simple classifiers: logistic regression, random forest and gradient boosting. The models use no tuning. They give a baseline, and they test if the preprocessed data can support a prediction model.

The notebook prints only summary statistics. It never prints patient rows.

**Before you run it:** run the pipeline (`scripts/run_all.sh`). The notebook reads `data/features/wide.parquet`, `splits.parquet` and `default_features.json`.

## 1. Setup and load

In [ ]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score, roc_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 42
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.yaml").exists()), None)
if ROOT is None:
    raise SystemExit("Start this notebook inside the project folder (the folder with config.yaml above it).")
FEAT = ROOT / "data" / "features"

wide = pd.read_parquet(FEAT / "wide.parquet")
splits = pd.read_parquet(FEAT / "splits.parquet")[["stay_id", "split_random", "split_temporal", "split_hybrid"]]
spec = json.loads((FEAT / "default_features.json").read_text())

df = wide.merge(splits, on="stay_id", validate="one_to_one")
LABEL, CAT = spec["label"], spec["categorical"]
FEATURES = spec["wide"]
NUM = [c for c in FEATURES if c not in CAT]
print(f"stays: {len(df):,}   patients: {df.subject_id.nunique():,}   suggested features: {len(FEATURES)}")

## 2. Data checks

These checks must pass before any model result has a meaning.

In [ ]:
FORBIDDEN_PREFIX = ("label_", "aki_by_", "onset_", "n_outlier_", "split_")
FORBIDDEN_NAMES = {"stay_id", "subject_id", "hadm_id", "intime", "time_zero", "anchor_year_group", "weight_src",
                   "hospital_expire_flag", "esrd_chronic_dialysis", "esrd_icd_current_adm", "ckd_icd_current_adm", "dialysis_hx_item"}

assert df.stay_id.is_unique, "stay_id must be unique"
assert set(FEATURES) <= set(df.columns), "a suggested feature is missing from the wide table"
assert not [c for c in FEATURES if c.startswith(FORBIDDEN_PREFIX)], "a label or outcome column is in the features"
assert not FORBIDDEN_NAMES & set(FEATURES), "an identifier or a Caution column is in the features"
for s in ["split_random", "split_temporal", "split_hybrid"]:
    assert (df.groupby("subject_id")[s].nunique() == 1).all(), f"a patient is in two parts of {s}"
assert df[LABEL].isin([0, 1, True, False]).all() and df[LABEL].notna().all()
print("all checks passed")

## 3. Exploratory analysis

### 3.1 Label and splits

In [ ]:
y_all = df[LABEL].astype(int)
print(f"label rate overall: {y_all.mean():.3f}  ({y_all.sum():,} of {len(df):,} stays)\n")
tab = (df.assign(y=y_all)
         .melt(id_vars="y", value_vars=["split_random", "split_temporal", "split_hybrid"], var_name="split", value_name="part")
         .groupby(["split", "part"]).y.agg(stays="size", positives="sum", rate="mean").round(3))
tab

### 3.2 Missing values

The pipeline keeps missing values as missing. This plot shows how much each feature is missing. Models must handle this.

In [ ]:
miss = df[NUM].isna().mean().sort_values(ascending=False)
top = miss.head(25)[::-1]
fig, ax = plt.subplots(figsize=(7, 6))
ax.barh(top.index, top.values * 100, color="#4c78a8")
ax.set_xlabel("missing values (%)")
ax.set_title("25 features with the most missing values")
plt.tight_layout(); plt.show()
print(f"features with any missing value: {(miss > 0).sum()} of {len(NUM)};  fully observed: {(miss == 0).sum()}")

### 3.3 Key features by label

The plots compare six clinically important features between label 0 and label 1. A feature that carries no signal would show equal boxes.

In [ ]:
KEY = ["creatinine_last", "bun_last", "urine_ml_kg_h_overall", "mbp_min", "hco3_min", "age_at_icu"]
KEY = [c for c in KEY if c in df.columns]
fig, axes = plt.subplots(2, 3, figsize=(12, 6))
for ax, c in zip(axes.ravel(), KEY):
    v = df[[c, LABEL]].dropna()
    lo, hi = v[c].quantile([0.01, 0.99])
    groups = [v.loc[v[LABEL].astype(int) == k, c].clip(lo, hi) for k in (0, 1)]
    ax.boxplot(groups, tick_labels=["no AKI", "AKI"], showfliers=False)
    ax.set_title(c, fontsize=10)
plt.tight_layout(); plt.show()
summary = df.assign(y=y_all).groupby("y")[KEY].median().round(2)
summary.index = ["no AKI (median)", "AKI (median)"]
summary

## 4. Models

- **Logistic regression**: median imputation, standard scaling, one-hot coding of the categorical columns, L2 penalty.
- **Random forest**: median imputation, one-hot coding.
- **Gradient boosting** (`HistGradientBoostingClassifier`): it reads missing values directly, so it needs no imputation.

All settings are fixed. The imputation and scaling are fitted on the train part only.

In [ ]:
X = df[FEATURES].copy()
bool_cols = X.select_dtypes("bool").columns
X[bool_cols] = X[bool_cols].astype(int)
y = y_all


def onehot():
    return OneHotEncoder(handle_unknown="ignore", min_frequency=20, sparse_output=False)


def make_models():
    lr_prep = ColumnTransformer([("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), NUM),
                                 ("cat", onehot(), CAT)])
    rf_prep = ColumnTransformer([("num", SimpleImputer(strategy="median"), NUM), ("cat", onehot(), CAT)])
    gb_prep = ColumnTransformer([("num", "passthrough", NUM), ("cat", onehot(), CAT)])
    return {
        "Logistic regression": Pipeline([("prep", lr_prep), ("clf", LogisticRegression(C=0.05, max_iter=3000))]),
        "Random forest": Pipeline([("prep", rf_prep), ("clf", RandomForestClassifier(
            n_estimators=300, min_samples_leaf=5, max_features="sqrt", class_weight="balanced_subsample", n_jobs=-1, random_state=SEED))]),
        "Gradient boosting": Pipeline([("prep", gb_prep), ("clf", HistGradientBoostingClassifier(
            max_iter=300, learning_rate=0.05, max_leaf_nodes=15, min_samples_leaf=40, l2_regularization=1.0,
            early_stopping=True, validation_fraction=0.15, n_iter_no_change=20, random_state=SEED))]),
    }


def evaluate(split_col, y_train=None):
    part = df[split_col]
    tr, va, te = (part == p for p in ("train", "val", "test"))
    rows, fitted, test_prob = [], {}, {}
    for name, model in make_models().items():
        model.fit(X[tr], (y[tr] if y_train is None else y_train))
        fitted[name] = model
        for label, m in (("val", va), ("test", te)):
            p = model.predict_proba(X[m])[:, 1]
            rows.append({"split": split_col, "model": name, "part": label, "AUROC": roc_auc_score(y[m], p),
                         "AUPRC": average_precision_score(y[m], p), "label rate": y[m].mean()})
            if label == "test": test_prob[name] = p
    return pd.DataFrame(rows), fitted, test_prob

### 4.1 Default split (`split_random`)

In [ ]:
res_random, fitted, test_prob = evaluate("split_random")
res_random.round(3).pivot(index="model", columns="part", values=["AUROC", "AUPRC"]).round(3)

In [ ]:
te = df["split_random"] == "test"
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
for name, p in test_prob.items():
    fpr, tpr, _ = roc_curve(y[te], p)
    ax[0].plot(fpr, tpr, label=f"{name} (AUROC {roc_auc_score(y[te], p):.3f})")
    prec_rec = pd.DataFrame({"p": p, "y": y[te].values}).sort_values("p", ascending=False)
    tp = prec_rec.y.cumsum(); k = np.arange(1, len(prec_rec) + 1)
    ax[1].plot(tp / tp.iloc[-1], tp / k, label=name)
ax[0].plot([0, 1], [0, 1], "k--", lw=0.8); ax[0].set_xlabel("false positive rate"); ax[0].set_ylabel("true positive rate"); ax[0].set_title("ROC curve (test)")
ax[1].axhline(y[te].mean(), color="k", ls="--", lw=0.8, label="label rate"); ax[1].set_xlabel("recall"); ax[1].set_ylabel("precision"); ax[1].set_title("Precision-recall curve (test)")
ax[0].legend(fontsize=8); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 5. Sanity checks on the data

### 5.1 Shuffled-label control

The control fits the same models on train labels in random order. The test AUROC must be close to 0.5. A higher value points to a leak or a bug in the code. The test labels stay true.

In [ ]:
rng = np.random.default_rng(SEED)
tr_mask = df["split_random"] == "train"
shuffled = pd.Series(rng.permutation(y[tr_mask].values), index=y[tr_mask].index)
ctrl, _, _ = evaluate("split_random", y_train=shuffled)
ctrl[ctrl.part == "test"][["model", "AUROC", "AUPRC", "label rate"]].round(3).reset_index(drop=True)

### 5.2 Which features drive the models

If the data are valid, the leading features must make clinical sense (kidney function, urine output, acid-base balance, illness severity). A leading feature that is an identifier or a time stamp would be a warning.

In [ ]:
gb = fitted["Gradient boosting"]
va = df["split_random"] == "val"
pi = permutation_importance(gb, X[va], y[va], scoring="roc_auc", n_repeats=5, random_state=SEED, n_jobs=-1)
imp = pd.Series(pi.importances_mean, index=X.columns).sort_values(ascending=False)

lr = fitted["Logistic regression"]
names = lr.named_steps["prep"].get_feature_names_out()
coef = pd.Series(lr.named_steps["clf"].coef_[0], index=[n.split("__", 1)[1] for n in names]).sort_values(key=np.abs, ascending=False)

fig, ax = plt.subplots(1, 2, figsize=(13, 5))
imp.head(15)[::-1].plot.barh(ax=ax[0], color="#4c78a8"); ax[0].set_title("Gradient boosting: permutation importance (val, AUROC drop)")
coef.head(15)[::-1].plot.barh(ax=ax[1], color="#f58518"); ax[1].set_title("Logistic regression: largest coefficients (scaled features)")
plt.tight_layout(); plt.show()

### 5.3 Other split definitions

The temporal split trains on early years and tests on the latest years. The label rate falls with time. A large drop against the random split would show time drift.

In [ ]:
res_all = pd.concat([res_random] + [evaluate(s)[0] for s in ("split_temporal", "split_hybrid")], ignore_index=True)
res_all[res_all.part == "test"].pivot(index="model", columns="split", values="AUROC").round(3)

### 5.4 Charting frequency

A column that ends in `_n_hours_obs` counts the hours with a value. It measures how often staff charted a value, not the state of the patient. The urine criterion needs frequent charting, so these counts can predict the label for a reason that is not clinical.

The table compares gradient boosting on different feature sets. If the score without the counts stays clearly above 0.5, the data carry clinical signal too.

In [ ]:
def gb_scores(columns):
    prep = ColumnTransformer([("num", "passthrough", [c for c in columns if c not in CAT]),
                              ("cat", onehot(), [c for c in columns if c in CAT])])
    model = Pipeline([("prep", prep), ("clf", make_models()["Gradient boosting"].named_steps["clf"])])
    tr, te = (df["split_random"] == p for p in ("train", "test"))
    model.fit(X.loc[tr, columns], y[tr])
    p = model.predict_proba(X.loc[te, columns])[:, 1]
    return {"features": len(columns), "AUROC": roc_auc_score(y[te], p), "AUPRC": average_precision_score(y[te], p)}


counts = [c for c in FEATURES if c.endswith("_n_hours_obs")]
urine = [c for c in FEATURES if c.startswith(("urine_ml", "fluid_balance"))]
variants = {
    "All suggested features": FEATURES,
    "Without the hours-observed counts": [c for c in FEATURES if c not in counts],
    "Without urine features": [c for c in FEATURES if c not in urine],
    "Hours-observed counts only": counts,
    "Creatinine, BUN and age only": [c for c in ["creatinine_last", "creatinine_max", "creatinine_mean", "bun_last", "bun_max", "age_at_icu", "creat_baseline_7d"] if c in FEATURES],
}
pd.DataFrame({k: gb_scores(v) for k, v in variants.items()}).T.round(3)

## 6. Summary

Read the results with these questions:

1. **Do the checks pass?** Section 2 must print "all checks passed".
2. **Is the signal real?** The test AUROC of the three models must be clearly above 0.5, and the shuffled-label control must be close to 0.5.
3. **Do the leading features make sense?** See sections 5.2 and 5.4.
4. **Is the model type important?** If the three models are close, the data limit the result, not the model. Tuning and better features can improve the baseline.
5. **Does the result hold on other splits?** See section 5.3.

If a check fails, do not use the results. Look at the pipeline steps first (see `docs/pipeline.md`).